# Multilingual Aligned Dataset
Builds a dataset where Latin-family and diverse-family language prefixes share a **common English suffix boundary**.
Both groups' cutoff sentence indices are computed independently, then the max is used as the shared boundary.

In [1]:
import asyncio
import hashlib
import json
import os
import re
import time

import nest_asyncio
import tiktoken
from datasets import Dataset, load_dataset
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAI
from tqdm.notebook import tqdm

nest_asyncio.apply()

OPENAI_MODEL = "gpt-5-mini-2025-08-07"
PREFIX_LENGTH = 50
SUFFIX_LENGTH = 50
MIN_TOKEN_LENGTH = 500
NUM_SEQUENCES = 2000
SEED = 42

LATIN_LANGUAGES = ["Spanish", "Portuguese", "French", "Italian"]
DIVERSE_LANGUAGES = ["Russian", "Chinese (Simplified)", "Tamil", "Arabic"]

In [2]:
def _split_sentences(text: str) -> list[str]:
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?]) +(?=[A-Z\"'])", text.strip())
        if s.strip()
    ]


async def _translate_batch(
    sentences: list[str], language: str, client, model: str
) -> list[str]:
    """Translate multiple sentences into a single language in one API call.

    Returns [translated_sent1, translated_sent2, ...]
    """
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    prompt = (
        f"Translate each of the following {len(sentences)} sentences into {language}.\n"
        f"Return ONLY a JSON object with a single key \"translations\" whose value is an array of translations "
        f"(one per sentence, in the same order).\n\n{numbered}"
    )
    max_retries = 8
    base_delay = 1.0
    for attempt in range(max_retries):
        try:
            response = await client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            result = json.loads(response.choices[0].message.content.strip())
            if "translations" not in result:
                raise ValueError("Missing 'translations' key in response")
            translations = result["translations"]
            if not isinstance(translations, list) or len(translations) != len(sentences):
                raise ValueError(
                    f"Expected {len(sentences)} translations, "
                    f"got {len(translations) if isinstance(translations, list) else type(translations)}"
                )
            return translations
        except (json.JSONDecodeError, ValueError) as e:
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"Batch parse error ({e}), retrying in {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                await asyncio.sleep(delay)
            else:
                raise
        except Exception as e:
            err = str(e).lower()
            if "rate" in err or "429" in err or "quota" in err or "capacity" in err:
                delay = base_delay * (2 ** attempt)
                print(f"Rate limited, retrying in {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                await asyncio.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries exceeded for batch translation to {language}")


def _load_cache(path: str) -> dict:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_cache(cache: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=4)

In [3]:
def _prepare_source_dataset(seed: int = SEED, min_token_length: int = MIN_TOKEN_LENGTH):
    """Load, filter, and shuffle the WikiText dataset."""
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
    pattern = r"(?: =)+[^=]+(?:= )+"
    dataset = dataset.filter(
        lambda x: not re.match(pattern, x["text"]),
        desc="Filtering headers",
    )
    dataset = dataset.shuffle(seed=seed)
    return dataset

In [4]:
def translate_all_sequences(
    sequences: list[dict],
    languages: list[str],
    cache: dict,
    cache_path: str,
    client,
    model_tag: str,
    encoding,
    prefix_length: int = PREFIX_LENGTH,
    max_batch_size: int = 50,
    max_concurrent: int = 8,
) -> tuple[dict[int, dict[str, list[int]]], dict[int, int]]:
    """Translate sentences round-by-round until every language has enough tokens.

    Each round collects sentence N from all active sequences, deduplicates,
    then translates uncached sentences per language in parallel API calls.

    Returns:
        prefix_ids_map: {seq_idx: {lang: [prefix_length token IDs]}}
        cutoff_map: {seq_idx: max cutoff sentence index across all languages}
    """
    model_name = model_tag.split(":")[-1]

    accumulated: dict[int, dict[str, list[int]]] = {
        s["seq_idx"]: {lang: [] for lang in languages} for s in sequences
    }
    cutoffs: dict[int, dict[str, int | None]] = {
        s["seq_idx"]: {lang: None for lang in languages} for s in sequences
    }
    seq_data = {s["seq_idx"]: s for s in sequences}
    active = {s["seq_idx"] for s in sequences}

    sent_round = 0
    api_calls = 0

    pbar = tqdm(total=len(sequences), desc="Translating sequences", unit="seq")

    async def _sem_translate(sem, sents, lang, cli, model):
        async with sem:
            return await _translate_batch(sents, lang, cli, model)

    async def _run():
        nonlocal sent_round, api_calls, active

        sem = asyncio.Semaphore(max_concurrent)

        while active:
            # Collect unique sentences needed this round
            round_sentences: dict[str, str] = {}  # hash -> sentence
            seq_hashes: dict[int, str] = {}  # seq_idx -> hash

            for idx in sorted(active):
                sents = seq_data[idx]["sentences"]
                if sent_round >= len(sents):
                    continue
                sent = sents[sent_round]
                h = hashlib.sha256(sent.encode("utf-8")).hexdigest()
                round_sentences[h] = sent
                seq_hashes[idx] = h

            # Collect all (lang, chunk) tasks for this round
            tasks = []
            task_meta = []  # (lang, chunk_hashes) per task

            for lang in languages:
                uncached = {
                    h: sent for h, sent in round_sentences.items()
                    if f"{model_tag}:{lang}:{h}" not in cache
                }

                if uncached:
                    sents_list = list(uncached.values())
                    hashes_list = list(uncached.keys())
                    for chunk_start in range(0, len(sents_list), max_batch_size):
                        chunk_sents = sents_list[chunk_start:chunk_start + max_batch_size]
                        chunk_hashes = hashes_list[chunk_start:chunk_start + max_batch_size]
                        tasks.append(_sem_translate(sem, chunk_sents, lang, client, model_name))
                        task_meta.append((lang, chunk_hashes))

            if tasks:
                results = await asyncio.gather(*tasks)
                api_calls += len(results)
                for (lang, chunk_hashes), translations in zip(task_meta, results):
                    for h, t in zip(chunk_hashes, translations):
                        cache[f"{model_tag}:{lang}:{h}"] = t
                _save_cache(cache, cache_path)

            # Accumulate tokens for all active sequences from cache
            for idx, h in seq_hashes.items():
                for lang in languages:
                    if cutoffs[idx][lang] is not None:
                        continue
                    translated = cache[f"{model_tag}:{lang}:{h}"]
                    accumulated[idx][lang].extend(encoding.encode(translated))
                    if len(accumulated[idx][lang]) >= prefix_length:
                        cutoffs[idx][lang] = sent_round

            # Drop completed or exhausted sequences
            done = {idx for idx in active if all(cutoffs[idx][lang] is not None for lang in languages)}
            exhausted = {
                idx for idx in active
                if sent_round >= len(seq_data[idx]["sentences"]) - 1 and idx not in done
            }
            active -= done | exhausted

            pbar.update(len(done))
            print(
                f"  Round {sent_round}: {api_calls} API calls total, "
                f"{len(done)} done, {len(active)} remaining"
            )
            sent_round += 1

    asyncio.run(_run())
    pbar.close()

    # Build return maps
    prefix_ids_map = {}
    cutoff_map = {}
    for s in sequences:
        idx = s["seq_idx"]
        if any(cutoffs[idx][lang] is None for lang in languages):
            continue
        prefix_ids_map[idx] = {
            lang: accumulated[idx][lang][:prefix_length] for lang in languages
        }
        cutoff_map[idx] = max(cutoffs[idx][lang] for lang in languages)

    print(
        f"  Complete: {len(prefix_ids_map)}/{len(sequences)} sequences successful, "
        f"{api_calls} API calls total"
    )
    return prefix_ids_map, cutoff_map

In [5]:
def process_multilingual_aligned(
    num_sequences: int = NUM_SEQUENCES,
    latin_cache_path: str = "cache_latin.json",
    diverse_cache_path: str = "cache_diverse.json",
    prefix_length: int = PREFIX_LENGTH,
    suffix_length: int = SUFFIX_LENGTH,
    min_token_length: int = MIN_TOKEN_LENGTH,
    seed: int = SEED,
    openai_model: str = OPENAI_MODEL,
    save_path_english: str = "wikitext_multilingual_english",
    save_path_latin: str = "wikitext_multilingual_latin",
    save_path_diverse: str = "wikitext_multilingual_diverse",
) -> Dataset:
    """Build a multilingual dataset with an aligned English suffix."""
    # Setup
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not found in environment / .env file")
    client = AsyncOpenAI(api_key=api_key)
    model_tag = f"openai:{openai_model}"
    encoding = tiktoken.get_encoding("o200k_base")

    # Load WikiText and collect valid sequences
    dataset = _prepare_source_dataset(seed=seed, min_token_length=min_token_length)

    sequences = []
    for example in tqdm(dataset, desc="Collecting source sequences", unit="doc"):
        if len(sequences) >= num_sequences:
            break
        tokens = encoding.encode(example["text"])
        if len(tokens) < min_token_length:
            continue
        full_text = encoding.decode(tokens)
        sents = _split_sentences(full_text)
        if len(sents) >= 2:
            sequences.append({
                "seq_idx": len(sequences),
                "sentences": sents,
                "tokens": tokens,
                "full_text": full_text,
            })

    print(f"Collected {len(sequences)} source sequences")

    # Translate: Latin languages
    latin_cache = _load_cache(latin_cache_path)
    print(f"\n--- Latin ({LATIN_LANGUAGES}) ---")
    latin_prefix_map, latin_cutoff_map = translate_all_sequences(
        sequences, LATIN_LANGUAGES, latin_cache, latin_cache_path,
        client, model_tag, encoding, prefix_length,
    )

    # Translate: Diverse languages
    diverse_cache = _load_cache(diverse_cache_path)
    print(f"\n--- Diverse ({DIVERSE_LANGUAGES}) ---")
    diverse_prefix_map, diverse_cutoff_map = translate_all_sequences(
        sequences, DIVERSE_LANGUAGES, diverse_cache, diverse_cache_path,
        client, model_tag, encoding, prefix_length,
    )

    # Pre-filter sequences and compute suffix IDs
    kept_sequences = []
    for s in sequences:
        idx = s["seq_idx"]
        if idx not in latin_prefix_map or idx not in diverse_prefix_map:
            continue
        shared_cutoff = max(latin_cutoff_map[idx], diverse_cutoff_map[idx])
        next_sent_idx = shared_cutoff + 1
        if next_sent_idx >= len(s["sentences"]):
            continue
        english_offset = sum(
            len(encoding.encode(s["sentences"][j])) for j in range(next_sent_idx)
        )
        suffix_ids = s["tokens"][english_offset : english_offset + suffix_length]
        if len(suffix_ids) < suffix_length:
            continue
        kept_sequences.append({
            "seq": s,
            "suffix_ids": suffix_ids,
            "kept_idx": len(kept_sequences),
        })

    print(f"\n{len(kept_sequences)} sequences kept, {len(sequences) - len(kept_sequences)} skipped.")

    if not kept_sequences:
        raise RuntimeError("No sequences survived assembly. Check PREFIX_LENGTH / data.")

    # Helper to build dataset rows for a language group
    def make_rows(kept, prefix_map, languages, group_name):
        rows = {"input_ids": [], "attention_mask": [], "language": [], "group": [], "source_idx": []}
        for k in kept:
            idx = k["seq"]["seq_idx"]
            for lang in languages:
                ids = list(prefix_map[idx][lang]) + list(k["suffix_ids"])
                rows["input_ids"].append(ids)
                rows["attention_mask"].append([1] * len(ids))
                rows["language"].append(lang)
                rows["group"].append(group_name)
                rows["source_idx"].append(k["kept_idx"])
        return rows

    # English prefix map (from original tokens, no translation needed)
    english_prefix_map = {
        k["seq"]["seq_idx"]: {"English": list(k["seq"]["tokens"][:prefix_length])}
        for k in kept_sequences
    }

    english_rows = make_rows(kept_sequences, english_prefix_map, ["English"], "english")
    latin_rows = make_rows(kept_sequences, latin_prefix_map, LATIN_LANGUAGES, "latin")
    diverse_rows = make_rows(kept_sequences, diverse_prefix_map, DIVERSE_LANGUAGES, "diverse")

    # Build and save datasets
    ds_english = Dataset.from_dict(english_rows).shuffle(seed=seed)
    ds_latin = Dataset.from_dict(latin_rows).shuffle(seed=seed)
    ds_diverse = Dataset.from_dict(diverse_rows).shuffle(seed=seed)

    ds_english.save_to_disk(save_path_english)
    ds_latin.save_to_disk(save_path_latin)
    ds_diverse.save_to_disk(save_path_diverse)
    print(f"English: {len(ds_english)} rows -> '{save_path_english}'")
    print(f"Latin:   {len(ds_latin)} rows -> '{save_path_latin}'")
    print(f"Diverse: {len(ds_diverse)} rows -> '{save_path_diverse}'")

In [6]:
ds = process_multilingual_aligned(num_sequences=NUM_SEQUENCES)

Collected 1861 source sequences

--- Latin (['Spanish', 'Portuguese', 'French', 'Italian']) ---


Translating sequences:   0%|          | 0/1861 [00:00<?, ?seq/s]

  Round 0: 0 API calls total, 443 done, 1418 remaining
  Round 1: 0 API calls total, 1111 done, 307 remaining
  Round 2: 0 API calls total, 274 done, 33 remaining
  Round 3: 0 API calls total, 31 done, 2 remaining
  Round 4: 0 API calls total, 1 done, 1 remaining
  Round 5: 0 API calls total, 0 done, 1 remaining
  Round 6: 0 API calls total, 1 done, 0 remaining
  Complete: 1861/1861 sequences successful, 0 API calls total

--- Diverse (['Russian', 'Chinese (Simplified)', 'Tamil', 'Arabic']) ---


Translating sequences:   0%|          | 0/1861 [00:00<?, ?seq/s]

  Round 0: 60 API calls total, 473 done, 1388 remaining
Batch parse error (Expected 50 translations, got 53), retrying in 1.0s (attempt 1/8)
  Round 1: 105 API calls total, 1127 done, 261 remaining
  Round 2: 116 API calls total, 240 done, 21 remaining
  Round 3: 120 API calls total, 17 done, 4 remaining
  Round 4: 123 API calls total, 3 done, 1 remaining
  Round 5: 127 API calls total, 0 done, 1 remaining
  Round 6: 131 API calls total, 1 done, 0 remaining
  Complete: 1861/1861 sequences successful, 131 API calls total

1859 sequences kept, 2 skipped.


Saving the dataset (0/1 shards):   0%|          | 0/1859 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7436 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7436 [00:00<?, ? examples/s]

English: 1859 rows -> 'wikitext_multilingual_english'
Latin:   7436 rows -> 'wikitext_multilingual_latin'
Diverse: 7436 rows -> 'wikitext_multilingual_diverse'


In [2]:
# ── Validation ──────────────────────────────────────────────────────────────
from collections import defaultdict
from datasets import load_from_disk

ds_eng = load_from_disk("wikitext_multilingual_english")
ds_lat = load_from_disk("wikitext_multilingual_latin")
ds_div = load_from_disk("wikitext_multilingual_diverse")

# Group rows by source_idx
def group_by_source(ds):
    groups = defaultdict(list)
    for row in ds:
        groups[row["source_idx"]].append(row)
    return groups

eng_groups = group_by_source(ds_eng)
lat_groups = group_by_source(ds_lat)
div_groups = group_by_source(ds_div)

all_source_idxs = sorted(eng_groups.keys() | lat_groups.keys() | div_groups.keys())

# 1. All languages present per source_idx
for idx in all_source_idxs:
    eng_langs = {r["language"] for r in eng_groups[idx]}
    lat_langs = {r["language"] for r in lat_groups[idx]}
    div_langs = {r["language"] for r in div_groups[idx]}
    assert eng_langs == {"English"}, f"source_idx {idx}: English missing, got {eng_langs}"
    assert lat_langs == set(LATIN_LANGUAGES), f"source_idx {idx}: Latin langs {lat_langs} != {set(LATIN_LANGUAGES)}"
    assert div_langs == set(DIVERSE_LANGUAGES), f"source_idx {idx}: Diverse langs {div_langs} != {set(DIVERSE_LANGUAGES)}"
print(f"[PASS] All languages present for each of {len(all_source_idxs)} source sequences.")

# 2. Shared suffix (last SUFFIX_LENGTH tokens identical across all rows for each source_idx)
for idx in all_source_idxs:
    all_rows = eng_groups[idx] + lat_groups[idx] + div_groups[idx]
    suffixes = [tuple(r["input_ids"][-SUFFIX_LENGTH:]) for r in all_rows]
    assert len(set(suffixes)) == 1, (
        f"source_idx {idx}: found {len(set(suffixes))} distinct suffixes across "
        f"{len(all_rows)} rows (expected 1)"
    )
print(f"[PASS] Shared suffix ({SUFFIX_LENGTH} tokens) verified for all source sequences.")

# 3. Fixed sequence length
for ds_name, ds in [("english", ds_eng), ("latin", ds_lat), ("diverse", ds_div)]:
    expected_len = PREFIX_LENGTH + SUFFIX_LENGTH
    for i, row in enumerate(ds):
        actual_len = len(row["input_ids"])
        assert actual_len == expected_len, (
            f"{ds_name} row {i}: length {actual_len} != expected {expected_len}"
        )
print(f"[PASS] All rows have length {PREFIX_LENGTH + SUFFIX_LENGTH} (prefix {PREFIX_LENGTH} + suffix {SUFFIX_LENGTH}).")

print(f"\nAll validations passed. "
      f"({len(ds_eng)} english, {len(ds_lat)} latin, {len(ds_div)} diverse rows)")

[PASS] All languages present for each of 1859 source sequences.
[PASS] Shared suffix (50 tokens) verified for all source sequences.
[PASS] All rows have length 100 (prefix 50 + suffix 50).

All validations passed. (1859 english, 7436 latin, 7436 diverse rows)


In [3]:
# ── Example: inspect all rows for a given source_idx ───────────────────────
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")
EXAMPLE_IDX = 2  # change this to inspect a different source sequence

print(f"=== source_idx = {EXAMPLE_IDX} ===\n")
for group_name, groups in [("English", eng_groups), ("Latin", lat_groups), ("Diverse", div_groups)]:
    print(f"── {group_name} ──")
    for row in sorted(groups[EXAMPLE_IDX], key=lambda r: r["language"]):
        prefix_tokens = row["input_ids"][:PREFIX_LENGTH]
        suffix_tokens = row["input_ids"][PREFIX_LENGTH:]
        prefix_text = encoding.decode(prefix_tokens)
        suffix_text = encoding.decode(suffix_tokens)
        print(f"  [{row['language']}]")
        print(f"    prefix: {prefix_text[:120]}...")
        print(f"    suffix: {suffix_text[:120]}...")
        print()

=== source_idx = 2 ===

── English ──
  [English]
    prefix:  On 10 May 1940 , Germany launched Fall Gelb , an invasion of the Low Countries . By attacking through the Netherlands ,...
    suffix:  To gain access to northern France , German forces would have to defeat the armed forces of the Low Countries and either...

── Latin ──
  [French]
    prefix: Le 10 mai 1940, l'Allemagne lança l'opération Fall Gelb, une invasion des Pays-Bas.En attaquant à travers les Pays-Bas, ...
    suffix:  To gain access to northern France , German forces would have to defeat the armed forces of the Low Countries and either...

  [Italian]
    prefix: Il 10 maggio 1940, la Germania lanciò l'operazione Fall Gelb, un'invasione dei Paesi Bassi.Attaccando attraverso i Paesi...
    suffix:  To gain access to northern France , German forces would have to defeat the armed forces of the Low Countries and either...

  [Portuguese]
    prefix: Em 10 de maio de 1940, a Alemanha lançou a operação Fall Gelb, uma in